In [1]:
import pandas as pd
import numpy as np
import pickle
import re

from sklearn.feature_extraction.text import TfidfVectorizer

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to C:\Users\Abarna
[nltk_data]     Studio\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Abarna
[nltk_data]     Studio\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Abarna
[nltk_data]     Studio\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## load dataset

In [2]:
faq = pd.read_csv("../data/merged_customer_support_dataset.csv")
faq.head()

,category,question,answer,source,id
0,Order Tracking,Can I track multiple orders?,"Yes, each order has its own tracking informati...",faq,NaN
1,Order Tracking,Why hasn't my order shipped?,Orders are usually processed within 24 hours. ...,faq,NaN
2,Order Tracking,Where is my order?,You can track your order from the 'My Orders' ...,faq,NaN
3,Order Tracking,Can I track multiple orders?,"Yes, each order has its own tracking informati...",faq,NaN
4,Order Tracking,Why hasn't my order shipped?,Orders are usually processed within 24 hours. ...,faq,NaN


## Dataset info

In [3]:
print("Dataset Shape:", faq.shape)

print("\nColumns:")
print(faq.columns.tolist())

print("\nCategories:")
print(faq["category"].unique())

Dataset Shape: (524, 5)

Columns:
['category', 'question', 'answer', 'source', 'id']

Categories:
['Order Tracking' 'Returns' 'Refunds' 'Shipping' 'Payments'
 'Technical Support' 'Warranty' 'Account' 'Membership'
 'Product Information' 'Policy' 'Product/Payment' 'Troubleshooting'
 'Billing' 'Contact']


## NLP Text cleaning

In [4]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z ]", " ", text)
    words = text.split()
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    return " ".join(words)

## Clean questions

In [5]:
faq["clean_question"] = faq["question"].apply(clean_text)
faq[["question","clean_question"]].head()

,question,clean_question
0,Can I track multiple orders?,track multiple order
1,Why hasn't my order shipped?,order shipped
2,Where is my order?,order
3,Can I track multiple orders?,track multiple order
4,Why hasn't my order shipped?,order shipped


## TF-IDF

In [6]:
vectorizer = TfidfVectorizer(max_features=300)
tfidf = vectorizer.fit_transform(faq["clean_question"])
keywords = vectorizer.get_feature_names_out()
print("Total Keywords:", len(keywords))
keywords[:50]

Total Keywords: 300


array(['accepted', 'accessible', 'accessory', 'accidental', 'account',
       'accurate', 'address', 'affirm', 'ap', 'api', 'app', 'appliance',
       'assistance', 'associated', 'attempt', 'availability', 'available',
       'base', 'benefit', 'bolt', 'call', 'cam', 'camera', 'cancel',
       'canceled', 'cancellation', 'cash', 'change', 'check', 'circ',
       'claim', 'cleaning', 'clear', 'clm', 'cloud', 'code', 'com',
       'compare', 'confirmation', 'connect', 'connection', 'contact',
       'covered', 'credit', 'damage', 'data', 'day', 'dedicated', 'deh',
       'dehumidifier'], dtype=object)

## Category keywords

In [7]:
category_keywords = {}
for category in faq["category"].unique():
    subset = faq[faq["category"] == category]
    vectorizer = TfidfVectorizer(max_features=15)
    X = vectorizer.fit_transform(subset["clean_question"])
    category_keywords[category] = vectorizer.get_feature_names_out().tolist()

In [8]:
for cat, words in category_keywords.items():
    print(cat)
    print(words)
    print("-"*50)

Order Tracking
['mean', 'multiple', 'order', 'shipment', 'shipped', 'track', 'transit']
--------------------------------------------------
Returns
['free', 'inspected', 'item', 'long', 'product', 'return', 'shipping', 'used']
--------------------------------------------------
Refunds
['cash', 'check', 'delayed', 'get', 'receive', 'refund', 'sent', 'status']
--------------------------------------------------
Shipping
['address', 'change', 'delayed', 'delivery', 'express', 'internationally', 'long', 'offer', 'package', 'ship', 'shipping', 'take']
--------------------------------------------------
Payments
['accepted', 'fail', 'method', 'online', 'pay', 'payment', 'secure', 'split', 'upi']
--------------------------------------------------
Technical Support
['assistance', 'contact', 'device', 'find', 'get', 'issue', 'remote', 'report', 'step', 'support', 'technical', 'troubleshooting', 'turn']
--------------------------------------------------
Warranty
['accidental', 'claim', 'covered', '

## Domain Synonyms

In [9]:
extra_synonyms = {
    "Refunds":[
        "refund",
        "money back",
        "reimbursement"
    ],
    "Returns":[
        "return",
        "replacement",
        "exchange"
    ],
    "Order Tracking":[
        "track",
        "tracking",
        "shipment",
        "delivery"
    ],
    "Payments":[
        "payment",
        "upi",
        "credit card",
        "wallet"
    ],
    "Shipping":[
        "shipping",
        "dispatch",
        "courier"
    ],
    "Account":[
        "login",
        "password",
        "profile"
    ],
    "Membership":[
        "membership",
        "subscription",
        "premium"
    ],
    "Warranty":[
        "repair",
        "guarantee"
    ],
    "Technical Support":[
        "technical",
        "issue",
        "bug",
        "support"
    ],
    "Product Information":[
        "product",
        "feature",
        "manual",
        "specification"
    ]
}

In [10]:
for category in category_keywords:

    if category in extra_synonyms:

        category_keywords[category].extend(
            extra_synonyms[category]
        )

    category_keywords[category] = list(
        set(category_keywords[category])
    )

In [11]:
question_examples = {}

for category in faq["category"].unique():

    question_examples[category] = faq[
        faq["category"] == category
    ]["question"].tolist()

In [12]:
nlp_pipeline = {

    "categories": faq["category"].unique().tolist(),

    "category_keywords": category_keywords,

    "question_examples": question_examples

}

In [13]:
with open("../data/nlp_pipeline.pkl","wb") as f:

    pickle.dump(nlp_pipeline,f)

print("NLP Pipeline Saved Successfully")

NLP Pipeline Saved Successfully


In [14]:
with open("../data/nlp_pipeline.pkl","rb") as f:

    nlp = pickle.load(f)

print(nlp.keys())

dict_keys(['categories', 'category_keywords', 'question_examples'])
